# 03. Feature Engineering & Preprocessing — EMIPredict AI

This notebook computes derived financial ratios (`debt_to_income_ratio`, `expense_to_income_ratio`, `affordability_ratio`, `risk_score`, `salary_credit_interaction`, `surplus_to_requested_ratio`), fits `OneHotEncoder` on categorical fields (including `company_type`), fits `StandardScaler` on numerical features, saves preprocessor binaries (`encoders.pkl`, `scaler.pkl`) to `models/preprocessing/`, and exports `engineered_dataset.csv` with all 48 features ready for model training.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load cleaned dataset
data_path = '../data/processed/cleaned_dataset.csv' if os.path.exists('../data') else 'data/processed/cleaned_dataset.csv'
df = pd.read_csv(data_path)
print(f"Loaded dataset for Feature Engineering: {df.shape[0]:,} rows")

In [ ]:
# 1. Financial Ratio Engineering (Shared canonical formulas)
total_obligations = (
    df['monthly_rent'] + df['current_emi_amount'] + df['school_fees'] +
    df['college_fees'] + df['travel_expenses'] + df['groceries_utilities'] +
    df['other_monthly_expenses']
)

df['debt_to_income_ratio'] = np.where(
    df['monthly_salary'] > 0,
    (df['current_emi_amount'] + df['monthly_rent']) / df['monthly_salary'],
    1.0
)

df['expense_to_income_ratio'] = np.where(
    df['monthly_salary'] > 0,
    total_obligations / df['monthly_salary'],
    1.0
)

surplus = np.maximum(0.0, df['monthly_salary'] - total_obligations)
implied_emi = np.where(df['requested_tenure'] > 0, df['requested_amount'] / df['requested_tenure'], 1.0)
df['affordability_ratio'] = np.where(implied_emi > 0, surplus / implied_emi, 0.0)

# Composite Risk Score (0 - 100)
norm_credit = (df['credit_score'] - 300) / 550.0
emp_weight = np.where(df['employment_type'] == 'Government', 1.0, np.where(df['employment_type'] == 'Private', 0.85, 0.70))
df['risk_score'] = np.clip((norm_credit * 50) + (np.minimum(df['years_of_employment'] / 10.0, 1.0) * 25) + (emp_weight * 25), 0.0, 100.0)

# Interaction Features
df['salary_credit_interaction'] = df['monthly_salary'] * (df['credit_score'] / 850.0)
df['surplus_to_requested_ratio'] = np.where(df['requested_amount'] > 0, surplus / df['requested_amount'], 0.0)

print("Financial ratios and interaction features engineered successfully.")

In [ ]:
# 2. Fit Preprocessors on Train Split Only
train_mask = df['dataset_split'] == 'train'
train_df = df[train_mask]

cat_cols = ['gender', 'marital_status', 'education', 'employment_type', 'company_type', 'house_type', 'emi_scenario']
num_cols = [
    'age', 'monthly_salary', 'years_of_employment', 'monthly_rent', 'family_size',
    'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities',
    'other_monthly_expenses', 'existing_loans', 'current_emi_amount', 'credit_score',
    'bank_balance', 'emergency_fund', 'requested_amount', 'requested_tenure',
    'debt_to_income_ratio', 'expense_to_income_ratio', 'affordability_ratio', 'risk_score',
    'salary_credit_interaction', 'surplus_to_requested_ratio'
]

# Fit & Apply One-Hot Encoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(train_df[cat_cols])
encoded_feature_names = list(encoder.get_feature_names_out(cat_cols))

encoded_cats = encoder.transform(df[cat_cols])
encoded_cat_df = pd.DataFrame(encoded_cats, columns=encoded_feature_names, index=df.index)

# Fit Standard Scaler on numerical features
scaler = StandardScaler()
scaler.fit(train_df[num_cols])

scaled_nums = scaler.transform(df[num_cols])
scaled_num_df = pd.DataFrame(scaled_nums, columns=num_cols, index=df.index)

# Save Preprocessors
os.makedirs('../models/preprocessing', exist_ok=True)
os.makedirs('models/preprocessing', exist_ok=True)
prep_dir = '../models/preprocessing' if os.path.exists('../models') else 'models/preprocessing'

joblib.dump(encoder, os.path.join(prep_dir, 'encoders.pkl'))
joblib.dump(scaler, os.path.join(prep_dir, 'scaler.pkl'))
print(f"Encoders and Scaler saved to {prep_dir}")

In [ ]:
# 3. Join Scaled Numerical & One-Hot Encoded Features & Export Engineered Dataset
full_engineered_df = pd.concat([scaled_num_df, encoded_cat_df, df[['emi_eligibility', 'max_monthly_emi', 'dataset_split']]], axis=1)

all_feature_cols = num_cols + encoded_feature_names
print(f"Total features engineered: {len(all_feature_cols)} ({len(num_cols)} numerical/interaction + {len(encoded_feature_names)} one-hot categorical)")

out_path = '../data/processed/engineered_dataset.csv' if os.path.exists('../data') else 'data/processed/engineered_dataset.csv'
full_engineered_df.to_csv(out_path, index=False)
print(f"Engineered dataset saved to {out_path} with shape {full_engineered_df.shape}")

## Final Feature Order (48 Features)

The 48 features passed to ML models follow this exact ordering:

### Scaled Numerical & Derived Ratios (24 features):
1. `age`
2. `monthly_salary`
3. `years_of_employment`
4. `monthly_rent`
5. `family_size`
6. `dependents`
7. `school_fees`
8. `college_fees`
9. `travel_expenses`
10. `groceries_utilities`
11. `other_monthly_expenses`
12. `existing_loans`
13. `current_emi_amount`
14. `credit_score`
15. `bank_balance`
16. `emergency_fund`
17. `requested_amount`
18. `requested_tenure`
19. `debt_to_income_ratio`
20. `expense_to_income_ratio`
21. `affordability_ratio`
22. `risk_score`
23. `salary_credit_interaction`
24. `surplus_to_requested_ratio`

### One-Hot Categorical Features (24 features):
25. `gender_Female`
26. `gender_Male`
27. `marital_status_Married`
28. `marital_status_Single`
29. `education_Graduate`
30. `education_High School`
31. `education_Post Graduate`
32. `education_Professional`
33. `employment_type_Government`
34. `employment_type_Private`
35. `employment_type_Self-employed`
36. `company_type_Enterprise`
37. `company_type_Government`
38. `company_type_Local`
39. `company_type_MNC`
40. `company_type_Startup`
41. `house_type_Family`
42. `house_type_Own`
43. `house_type_Rented`
44. `emi_scenario_E-commerce Shopping`
45. `emi_scenario_Education`
46. `emi_scenario_Home Appliances`
47. `emi_scenario_Personal Loan`
48. `emi_scenario_Vehicle`

*This order matches `classifier_metadata.json`, `regressor_metadata.json`, and `scripts/feature_utils.py`.*